In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

## Model Training Preparation

In [ ]:
import os
import shutil

SOURCE_TRAIN_DIR = "/kaggle/input/datasets/oyeyemidareazeez/training-data"
WORKING_TRAIN_DIR = "/kaggle/working/train_data"

if os.path.exists(WORKING_TRAIN_DIR):
    shutil.rmtree(WORKING_TRAIN_DIR)

shutil.copytree(SOURCE_TRAIN_DIR, WORKING_TRAIN_DIR)

print("Copied training data to:", WORKING_TRAIN_DIR)
print("Sample files:", os.listdir(WORKING_TRAIN_DIR)[:10])

In [ ]:
import os

TRAIN_DIR = "/kaggle/working/train_data/train_data"

files = sorted(os.listdir(TRAIN_DIR))

image_files = [
    f for f in files
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
]

txt_files = [
    f for f in files
    if f.lower().endswith(".txt")
]

print("Total files:", len(files))
print("Images:", len(image_files))
print("Captions:", len(txt_files))
print("Sample images:", image_files[:5])
print("Sample captions:", txt_files[:5])

## Creating metadata.json file

Containing a description (used as trigger words) for each image to perform training

In [ ]:
import os
import json

TRAIN_DIR = "/kaggle/working/train_data/train_data"
metadata_path = os.path.join(TRAIN_DIR, "metadata.jsonl")

image_extensions = (".png", ".jpg", ".jpeg", ".webp")

count = 0
missing_captions = []

with open(metadata_path, "w", encoding="utf-8") as outfile:
    for filename in sorted(os.listdir(TRAIN_DIR)):
        if not filename.lower().endswith(image_extensions):
            continue

        base_name = os.path.splitext(filename)[0]
        txt_path = os.path.join(TRAIN_DIR, f"{base_name}.txt")

        if not os.path.exists(txt_path):
            missing_captions.append(filename)
            continue

        with open(txt_path, "r", encoding="utf-8") as f:
            caption = f.read().strip()

        record = {
            "file_name": filename,
            "text": caption
        }

        outfile.write(json.dumps(record) + "\n")
        count += 1

print(f"metadata.jsonl created with {count} entries")
print("Saved at:", metadata_path)

if missing_captions:
    print("Images missing captions:", missing_captions[:10])

In [ ]:
%cd /kaggle/working

!pip install -q transformers accelerate peft safetensors datasets ftfy bitsandbytes

!rm -rf diffusers
!git clone https://github.com/huggingface/diffusers.git

%cd /kaggle/working/diffusers
!pip install -q -e .
!pip install -q -r examples/text_to_image/requirements.txt

%cd /kaggle/working

In [ ]:
!pip install -U xformers

## Setting Training parameters and fine-tuning the pre-trained SDXL model

saving the new weighted tensor to kaggle/working/facial_palsy_sdxl_lora_output

In [ ]:
!PYTHONPATH=/kaggle/working/diffusers/src accelerate launch \
  --num_processes=1 \
  --mixed_precision=bf16 \
  --num_machines=1 \
  --dynamo_backend=no \
  /kaggle/working/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="/kaggle/working/train_data/train_data" \
  --image_column="image" \
  --caption_column="text" \
  --resolution=512 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=10 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --rank=8 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --output_dir="/kaggle/working/facial_palsy_sdxl_lora_output" \
  --checkpointing_steps=500 \
  --seed=42

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/facial_palsy_sdxl_lora_output"

print(os.listdir(OUTPUT_DIR))

In [ ]:
import sys


for key in list(sys.modules.keys()):
    if key.startswith("diffusers"):
        del sys.modules[key]

sys.path.insert(0, "/kaggle/working/diffusers/src")

import diffusers

print("Diffusers file:", diffusers.__file__)
print("Diffusers version:", getattr(diffusers, "__version__", "NO VERSION FOUND"))

from diffusers import StableDiffusionXLPipeline

print("StableDiffusionXLPipeline imported successfully.")

In [ ]:
import sys
import torch
from IPython.display import display

for key in list(sys.modules.keys()):
    if key.startswith("diffusers"):
        del sys.modules[key]

sys.path.insert(0, "/kaggle/working/diffusers/src")

from diffusers import StableDiffusionXLPipeline

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
LORA_DIR = "/kaggle/working/facial_palsy_sdxl_lora_output"

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

pipe.load_lora_weights(LORA_DIR)
pipe.enable_attention_slicing()

print("SDXL LoRA loaded successfully.")

## Loading Model and Testing txt2Img prompt

In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

from diffusers import StableDiffusionXLPipeline

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
LORA_DIR = "/kaggle/input/datasets/oyeyemidareazeez/lora-sdxl/kaggle/working/facial_palsy_sdxl_lora_output/pytorch_lora_weights.safetensors"

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

pipe.load_lora_weights(LORA_DIR)
pipe.enable_attention_slicing()

print("SDXL LoRA loaded successfully.")

## Healthy

In [ ]:
prompt = (
    "fpalsy_healthy, clinical frontal face photo a man patient, "
        "healthy face, normal facial symmetry, level mouth corners, both eyes equally open, "
        "balanced eyebrows, neutral expression" )

negative_prompt = ( 
    "multiple faces, two people, group, collage, grid, text, watermark, "
    "side view, profile view, blurry, low quality, cartoon, painting, "
    "extra eyes, extra mouth, bad anatomy, "
    "wrong side palsy, bilateral palsy" ) 

generator = torch.Generator(device="cuda").manual_seed(68)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=512,
    height=512,
    num_inference_steps=45,
    guidance_scale=8.5,
    generator=generator
).images[0]

display(image)
image.save("/kaggle/working/test_severe_left.png")

In [ ]:
prompt = (
    "fpalsy_healthy, clinical frontal face photo of a lady patient, "
        "healthy face, normal facial symmetry, level mouth corners, both eyes equally open, "
        "balanced eyebrows, neutral expression" )

negative_prompt = ( 
    "multiple faces, two people, group, collage, grid, text, watermark, "
    "side view, profile view, blurry, low quality, cartoon, painting, "
    "extra eyes, extra mouth, bad anatomy, "
    "wrong side palsy, bilateral palsy" ) 

generator = torch.Generator(device="cuda").manual_seed(90)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=512,
    height=512,
    num_inference_steps=45,
    guidance_scale=8.5,
    generator=generator
).images[0]

display(image)
image.save("/kaggle/working/test_severe_left.png")

## Severe palsy

In [ ]:
prompt = (
    "fpalsy_severe_left, clinical frontal photo of a man , "
    "patient's right facial palsy, affected side is viewer-left, "
    "viewer-left mouth droop, eye closed, "
    "viewer-left eyebrow lower, severe unilateral asymmetry, "
    "neutral face, no smile, realistic medical photo" )

negative_prompt = ( 
    "multiple faces, two people, group, collage, grid, text, watermark, " 
    "side view, profile view, blurry, low quality, cartoon, painting, " 
    "extra eyes, extra mouth, bad anatomy, " 
    "viewer-right facial palsy, viewer-right mouth droop, viewer-right closed eye, " 
    "wrong side palsy, bilateral palsy" ) 

generator = torch.Generator(device="cuda").manual_seed(50)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=512,
    height=512,
    num_inference_steps=45,
    guidance_scale=8.5,
    generator=generator
).images[0]

display(image)
image.save("/kaggle/working/test_severe_left.png")

## Dataset Generation

In [ ]:
test_prompts = {
    "healthy": (
        "fpalsy_healthy, clinical frontal face photo of one lady patient, "
        "healthy face, normal facial symmetry, level mouth corners, both eyes equally open, "
        "balanced eyebrows, neutral expression"
    ),
    
    "mild_left": (
        "fpalsy_mild_right, clinical frontal face photo of asian lady, "
        "mild left facial palsy, slight left mouth droop, mild left eyelid weakness, "
        "slight eyebrow asymmetry, subtle facial asymmetry, neutral expression"
    ),

    "moderate_left": (
        "fpalsy_moderate_left, clinical frontal face photo of a man patient, "
        "moderate left facial palsy, visible left mouth droop, left eyelid partly closed, "
        "left eyebrow lower, clear facial asymmetry, neutral expression"
    ),

    "severe_left": (
        "fpalsy_severe_left, clinical frontal photo of an american man, "
        "severe left facial palsy, droop on the left side of the mouth, left eye closed, "
        "left eyebrow significantly lower, severe unilateral asymmetry, "
        "neutral face, no smile, realistic medical photo"
    ),

    "mild_right": (
        "fpalsy_mild_left, clinical frontal face photo of one teen woman, "
        "mild right facial palsy, slight right mouth droop, mild right eyelid weakness, "
        "slight eyebrow asymmetry, subtle facial asymmetry, neutral expression"
    ),

    "moderate_right": (
        "fpalsy_moderate_right, clinical frontal photo of one asian man , "
        "moderate right facial palsy, visible right mouth droop, right eyelid partly closed, "
        "right eyebrow lower, clear facial asymmetry, neutral expression"
    ),

    "severe_right": (
        "fpalsy_severe_right, clinical frontal photo of a black man, "
        "severe right facial palsy, droop on the right side of the mouth, right eye closed, "
        "right eyebrow significantly lower, severe unilateral asymmetry, "
        "neutral face, no smile, realistic medical photo"
    )
}

negative_prompt = (
    "multiple faces, two people, group, collage, grid, text, watermark, "
    "side view, profile view, blurry, low quality, cartoon, painting, "
    "extra eyes, extra mouth, bad anatomy, "
    "wrong side palsy, bilateral palsy"
)

for name, prompt in test_prompts.items():
    generator = torch.Generator(device="cuda").manual_seed(44)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=512,
        height=512,
        num_inference_steps=45,
        guidance_scale=8.5,
        generator=generator
    ).images[0]

    display(image)
    print("Filename:", f"sdxl_test_{name}.png")

## Dataset Generation (Diverse seed step 3)

In [ ]:
import random
import itertools


BASE_SEED = 44
NUM_SEEDS = 20
SEED_STEP = 3


OUTPUT_DIR = "/kaggle/working/facial_palsy_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

genders = ["man", "woman"]
ethnicities = ["black", "asian", "indian", "arab", "white"]
ages = ["teen", "adult", "old"]

demographics = [
    f"a {age} {eth} {gender}"
    for gender, eth, age in itertools.product(genders, ethnicities, ages)
]


palsy_templates = {
    "healthy": (
        "fpalsy_healthy, clinical frontal face photo of {subject}, "
        "healthy face, normal facial symmetry, level mouth corners, both eyes equally open, "
        "balanced eyebrows, neutral expression"
    ),
    "mild_left": (
        "fpalsy_mild_right, clinical frontal face photo of {subject}, "
        "mild left facial palsy, slight left mouth droop, mild left eyelid weakness, "
        "slight eyebrow asymmetry, subtle facial asymmetry, neutral expression"
    ),
    "moderate_left": (
        "fpalsy_moderate_left, clinical frontal face photo of {subject}, "
        "moderate left facial palsy, visible left mouth droop, left eyelid partly closed, "
        "left eyebrow lower, clear facial asymmetry, neutral expression"
    ),
    "severe_left": (
        "fpalsy_severe_left, clinical frontal photo of {subject}, "
        "severe left facial palsy, droop on the left side of the mouth, left eye closed, "
        "left eyebrow significantly lower, severe unilateral asymmetry, "
        "neutral face, no smile, realistic medical photo"
    ),
    "mild_right": (
        "fpalsy_mild_left, clinical frontal face photo of {subject}, "
        "mild right facial palsy, slight right mouth droop, mild right eyelid weakness, "
        "slight eyebrow asymmetry, subtle facial asymmetry, neutral expression"
    ),
    "moderate_right": (
        "fpalsy_moderate_right, clinical frontal photo of {subject}, "
        "moderate right facial palsy, visible right mouth droop, right eyelid partly closed, "
        "right eyebrow lower, clear facial asymmetry, neutral expression"
    ),
    "severe_right": (
        "fpalsy_severe_right, clinical frontal photo of {subject}, "
        "severe right facial palsy, droop on the right side of the mouth, right eye closed, "
        "right eyebrow significantly lower, severe unilateral asymmetry, "
        "neutral face, no smile, realistic medical photo"
    )
}

negative_prompt = (
    "multiple faces, two people, group, collage, grid, text, watermark, "
    "side view, profile view, blurry, low quality, cartoon, painting, "
    "extra eyes, extra mouth, bad anatomy, deformed face, "
    "wrong side palsy, duplicated features, bilateral palsy"
)

class_keys = list(palsy_templates.keys())

for seed_offset in range(NUM_SEEDS):

    seed = BASE_SEED + seed_offset * SEED_STEP

    print(f"\n--- SEED {seed} ---")

    for class_name in class_keys:
        
        rng = random.Random(seed + hash(class_name))

        shuffled_demo = demographics.copy()
        rng.shuffle(shuffled_demo)

        class_dir = os.path.join(OUTPUT_DIR, class_name)
        os.makedirs(class_dir, exist_ok=True)

        
        for i, subject in enumerate(shuffled_demo):

            generator = torch.Generator(device="cuda").manual_seed(seed)

            prompt = palsy_templates[class_name].format(subject=subject)

            image = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                width=512,
                height=512,
                num_inference_steps=45,
                guidance_scale=8.5,
                generator=generator
            ).images[0]

            print(f"Seed {seed} | Class {class_name} | {i+1}/{len(shuffled_demo)} | {subject}")

            filename = f"{class_name}_seed{seed}_img{i}.png"
            image.save(os.path.join(class_dir, filename))

print("\nDONE")

### Zipping Saved Dataset

In [ ]:
import shutil

folder_to_zip = "/kaggle/working/facial_palsy_dataset"


output_zip_name = "/kaggle/working/facial_palsy_dataset"

print("\n== STARTING COMPRESSION ==")
print(f"Zipping folder: {folder_to_zip} ...")

zipped_file_path = shutil.make_archive(output_zip_name, 'zip', folder_to_zip)

print(f"Successfully created archive at: {zipped_file_path}")